In [115]:
import pandas as pd
import numpy as np
import datetime
import json
import os
from IPython.display import display_html
from IPython.display import HTML
import csv

In [116]:
def display_horizontal(*args):
    html_str = ''
    for df in args:
        html_str += df.to_html()
    display_html(
        html_str.replace('table','table style="display:inline;margin-right:50px"'), 
        raw=True
    )

In [117]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)
    
with open("INDEX_NAME.json", 'r', encoding = 'utf-8') as f:
    INDEX_NAME = json.load(f)

In [118]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [119]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)

In [120]:
UNIVERSE_A = {}
UNIVERSE_O = {}
target_dates = {}

for ticker in INDEX_DATA:
    if ticker in a_share_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_A[ticker] = INDEX_DATA[ticker]

    elif ticker in other_market_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_O[ticker] = INDEX_DATA[ticker]

In [121]:
print(f"Assets under Analysis: {len(UNIVERSE_A)}  {len(UNIVERSE_O)}")
print(f"Total Assets: {len(INDEX_DATA)}")

Assets under Analysis: 73  35
Total Assets: 186


In [122]:
n = 0
for ticker in other_market_dict:
    if ticker in INDEX_DATA:
        n += 1

n

39

## Z Score Calculation

In [126]:
UNIVERSE = UNIVERSE_A
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 5
duration = years * 250

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [127]:
display_number = 20

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', '市盈率'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', '股息率'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='市盈率', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='股息率', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "市盈率"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "股息率"]]

In [128]:
print()
print("A股")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


A股
2025-05-15
5年


,Ticker,指数名称,市盈率
9,399812.SZ,养老产业,-1.681620
37,h30035.CSI,300非银,-1.521051
58,399966.SZ,中证800证保,-1.510972
47,931068.CSI,消费龙头,-1.264988
45,931139.CSI,CS消费50,-1.193538
50,930653.CSI,CS食品饮,-1.188510
10,000815.CSI,细分食品,-1.153720
7,707717L.MI,MSCI中国A股质量,-1.070096
46,931079.CSI,5G通信,-0.999484
49,931008.CSI,汽车指数,-0.895608


In [141]:
UNIVERSE = UNIVERSE_O
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 1
duration = years * 252

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_35412\379741716.py:20: RuntimeWarning: invalid value encountered in scalar divide
  Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
C:\Users\Bill Yin\AppData\Local\Temp\ipykernel_35412\379741716.py:33: RuntimeWarning: invalid value encountered in scalar divide
  Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std


In [142]:
display_number = 20

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', '市盈率'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', '股息率'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='市盈率', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='股息率', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "市盈率"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "股息率"]]

In [143]:
print()
print("海外")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


海外
2025-05-15
1年


,Ticker,指数名称,市盈率
10,931787CNY00.CSI,港股创新药,-1.359463
12,750108.MI,美国50,-1.084318
0,NDX.GI,纳斯达克指数,-0.847095
7,N225.GI,日经225,-0.827758
5,NDXTMC.GI,纳指科技,-0.819942
31,987008.CNI,港股通科技30,-0.818967
24,SPTR500N.SPI,标普500,-0.658269
23,SPXEWTR.SPI,标普500等权,-0.658269
1,SPX.GI,标普500南方,-0.656705
27,931024.CSI,港股非银,-0.616471


In [114]:
ticker

'000913.SH'

In [110]:
temp = INDEX_DATA[ticker]

In [111]:
temp = temp[~temp.index.duplicated(keep = "last")]

In [113]:
temp.tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-05-09,28.068100,1.9990,7848.4305
2025-05-12,27.993900,1.8358,7818.1952
2025-05-13,28.240000,1.8197,7891.3630
2025-05-14,28.418400,1.8083,7959.5245
2025-05-15,28.409901,1.8086,7968.7475
